# 03 — Statistical Analysis

Establish a statistical baseline before any ML: which factors are actually associated with 30-day readmission, and how strongly? At n≈100K, hypothesis tests have enormous power — nearly everything comes back "significant". So alongside p-values we look at effect sizes (Cramér's V, odds ratios) to judge what's practically meaningful, not just detectable.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

df = pd.read_csv('../data/processed/diabetic_data_clean.csv')
df.shape

(100114, 48)

## 1. Categorical associations — chi-square test of independence

In [2]:
def cramers_v(chi2, n, r, k):
    return np.sqrt((chi2 / n) / (min(r - 1, k - 1)))

cat_cols = ['race', 'gender', 'A1Cresult', 'max_glu_serum', 'diabetesMed', 'change', 'admission_type_id']
rows = []
for col in cat_cols:
    ct = pd.crosstab(df[col], df['readmitted_30d'])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    v = cramers_v(chi2, ct.values.sum(), *ct.shape)
    rows.append({'feature': col, 'chi2': round(chi2, 1), 'dof': dof, 'p_value': p, 'cramers_v': round(v, 4)})
chi2_results = pd.DataFrame(rows).sort_values('cramers_v', ascending=False)
chi2_results

,feature,chi2,dof,p_value,cramers_v
4,diabetesMed,67.0,1,2.693943e-16,0.0259
2,A1Cresult,40.6,3,8.046436e-09,0.0201
5,change,33.3,1,7.703322e-09,0.0183
6,admission_type_id,31.7,7,4.684779e-05,0.0178
0,race,26.7,5,6.420584e-05,0.0163
3,max_glu_serum,19.4,3,2.273066e-04,0.0139
1,gender,1.1,2,5.822576e-01,0.0033


**Reading this:** every categorical here is p < 0.001 significant — not surprising at n≈100K. Cramér's V (0 = no association, 1 = perfect) is the number that matters, and every one of these is under 0.03 — by conventional benchmarks (Cohen), that's a *small-to-negligible* effect individually. `gender` in particular is not even statistically significant (p=0.58). None of these categorical demographic/medication-monitoring fields are strong standalone predictors — consistent with the EDA. They may still contribute in combination inside a multivariate model.

## 2. Numeric associations — Mann-Whitney U (distributions are skewed/count-like, not normal, so we don't use a t-test)

In [3]:
def rank_biserial(u, n1, n2):
    return 1 - (2 * u) / (n1 * n2)

num_cols = ['time_in_hospital', 'num_medications', 'number_inpatient', 'number_emergency',
            'number_outpatient', 'num_lab_procedures', 'number_diagnoses']
rows = []
g1_mask = df['readmitted_30d'] == 1
for col in num_cols:
    g0, g1 = df.loc[~g1_mask, col], df.loc[g1_mask, col]
    u, p = stats.mannwhitneyu(g1, g0, alternative='two-sided')
    effect = rank_biserial(u, len(g1), len(g0))
    rows.append({
        'feature': col, 'median_no_readmit': g0.median(), 'median_readmit_30d': g1.median(),
        'p_value': p, 'rank_biserial_effect': round(effect, 4),
    })
mwu_results = pd.DataFrame(rows).sort_values('rank_biserial_effect', key=abs, ascending=False)
mwu_results

,feature,median_no_readmit,median_readmit_30d,p_value,rank_biserial_effect
2,number_inpatient,0.0,0.0,0.000000e+00,-0.2142
0,time_in_hospital,4.0,4.0,3.522771e-60,-0.0933
1,num_medications,15.0,16.0,5.654621e-51,-0.0863
6,number_diagnoses,8.0,9.0,1.084514e-56,-0.0858
3,number_emergency,0.0,0.0,1.546184e-91,-0.0638
5,num_lab_procedures,44.0,45.0,1.124487e-13,-0.0427
4,number_outpatient,0.0,0.0,2.322595e-27,-0.0402


`number_inpatient` (prior inpatient visits) has by far the largest rank-biserial effect size of anything tested — consistent with the EDA finding and a strong preview of what SHAP will later confirm as the top model driver. Everything else is a much smaller, though still statistically real, effect.

## 3. Logistic regression baseline — odds ratios

In [4]:
baseline_features = [
    'time_in_hospital', 'num_medications', 'number_inpatient', 'number_emergency',
    'number_outpatient', 'number_diagnoses', 'num_lab_procedures',
]
X = df[baseline_features].astype(float)
X = sm.add_constant(X)
y = df['readmitted_30d']

logit_model = sm.Logit(y, X).fit(disp=0)
print(logit_model.summary())

                           Logit Regression Results                           
Dep. Variable:         readmitted_30d   No. Observations:               100114
Model:                          Logit   Df Residuals:                   100106
Method:                           MLE   Df Model:                            7
Date:                Tue, 25 Aug 2026   Pseudo R-squ.:                 0.03330
Time:                        19:20:27   Log-Likelihood:                -34226.
converged:                       True   LL-Null:                       -35405.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -2.9093      0.047    -61.566      0.000      -3.002      -2.817
time_in_hospital       0.0229      0.004      6.027      0.000       0.015       0.030
num_medications     

In [5]:
odds_ratios = pd.DataFrame({
    'coef': logit_model.params,
    'odds_ratio': np.exp(logit_model.params),
    'p_value': logit_model.pvalues,
}).drop('const').sort_values('odds_ratio', ascending=False)
odds_ratios

,coef,odds_ratio,p_value
number_inpatient,0.272084,1.312697,0.000000e+00
number_diagnoses,0.055026,1.056568,3.319703e-21
number_emergency,0.030000,1.030455,3.108005e-04
time_in_hospital,0.022866,1.023130,1.672636e-09
num_medications,0.004992,1.005005,4.331920e-04
num_lab_procedures,0.000528,1.000528,3.402345e-01
number_outpatient,-0.001675,0.998327,8.251556e-01


**Interpretation:** holding the other features fixed, each additional prior inpatient visit multiplies the odds of a 30-day readmission by roughly the `number_inpatient` odds ratio above — the single strongest multivariate driver, consistent with both the univariate effect-size ranking and the later SHAP analysis. This simple 7-feature logistic regression is the statistical floor the ML models in `05_model_training.ipynb` need to beat.

In [6]:
from sklearn.metrics import roc_auc_score
baseline_pred = logit_model.predict(X)
print('Baseline (7-feature) logistic regression ROC-AUC:', round(roc_auc_score(y, baseline_pred), 4))

Baseline (7-feature) logistic regression ROC-AUC:

 0.6309


## Summary

- Demographic/medication-monitoring categoricals (race, gender, A1Cresult, max_glu_serum, diabetesMed, change, admission_type) are statistically associated with readmission but each has a small effect size individually — `gender` is not significant at all.
- Prior utilization (`number_inpatient` especially) has by far the largest effect size of any single numeric feature, both by rank-biserial correlation and by multivariate odds ratio.
- A bare-bones 7-feature logistic regression already clears a meaningful ROC-AUC — the bar `04_feature_engineering.ipynb` → `05_model_training.ipynb` needs to raise using richer feature engineering and nonlinear models.